In [ ]:
import pandas as pd
import torch
import random
import numpy as np

from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import save_file, load_file
from transformers import AutoModel
from transformers import AutoTokenizer
from sentence_transformers.util import cos_sim
import contextlib
from tqdm import tqdm

In [3]:
my_seed = 511
random.seed(my_seed)
np.random.seed(my_seed)
gen = torch.Generator();
gen.manual_seed(my_seed);
torch.manual_seed(my_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(my_seed)
    torch.cuda.manual_seed_all(my_seed)

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
MODEL_NAME = "sberbank-ai/sbert_large_nlu_ru"

### metrics

In [8]:
TOP_K_METRICS = [1, 10]

In [9]:
def experiment(model):
    model.eval()
    sentences = [
        "Организация ремонта и технического обеспечения учреждений здравоохранения",
        "В нашем поликлиническом отделении уже несколько месяцев не проводится необходимый ремонт, просьба вмешаться.",
        "Прошу помочь в достижении стабильного энергообеспечения потребителей."
    ]
    embeddings = model.encode(sentences, convert_to_tensor=True)
    print(f'''
            pos: {torch.nn.functional.cosine_similarity(embeddings[0], embeddings[1], dim=0)},
            neg: {torch.nn.functional.cosine_similarity(embeddings[0], embeddings[2], dim=0)}''')

In [ ]:
def delta_sim(model, df):
    model.eval()
    emb_pos = model.encode(df['pos_text'].tolist(), convert_to_tensor=True)
    emb_easy_neg = model.encode(df['neg_text'].tolist(), convert_to_tensor=True)
    emb_hard_neg = model.encode(df['hard_neg'].tolist(), convert_to_tensor=True)

    sim_pos_easy = cos_sim(emb_pos, emb_easy_neg).diagonal().cpu()
    sim_pos_hard = cos_sim(emb_pos, emb_hard_neg).diagonal().cpu()
    sim_pos_pos = cos_sim(emb_pos, emb_pos).diagonal().cpu()

    delta_easy = sim_pos_pos - sim_pos_easy
    delta_hard = sim_pos_pos - sim_pos_hard

    print("Средняя delta similarity для easy negatives:", np.mean(delta_easy.numpy()))
    print("Средняя delta similarity для hard negatives:", np.mean(delta_hard.numpy()))

In [ ]:
def get_similarity_matrix(model, text, func):
    """
    Создает матрицу схожести: тексты × уникальные функции
    Размерность: (len(df['pos_text']), len(df['func'].unique()))
    """
    model.eval()
    embeddings_text = model.encode(text, convert_to_tensor=True)
    embeddings_func = model.encode(func, convert_to_tensor=True)
    sims = cos_sim(embeddings_text, embeddings_func)
    return sims.cpu().numpy()

def recall_at_k(sims, unique_funcs, true_funcs, k=3):
    '''(число правильных функций, найденных в top-K) / (все релевантные функции)'''
    top_k_indices = np.argsort(-sims, axis=1)[:, :k]
    hits = []

    for i, true_func in enumerate(true_funcs):
        top_k_funcs = [unique_funcs[idx] for idx in top_k_indices[i]]
        hits.append(true_func in top_k_funcs)

    return np.mean(hits)

def map_at_k(sims, unique_funcs, true_funcs, k=3):
    '''MAP@K - чувствителен к позиции правильного ответа'''
    top_k_indices = np.argsort(-sims, axis=1)[:, :k]
    ap_scores = []

    for i, true_func in enumerate(true_funcs):
        top_k_funcs = [unique_funcs[idx] for idx in top_k_indices[i]]
        hits = [1 if func == true_func else 0 for func in top_k_funcs]

        if sum(hits) == 0:
            ap_scores.append(0)
        else:
            precisions = []
            for idx, hit in enumerate(hits):
                if hit == 1:
                    precision_at_rank = sum(hits[:idx+1]) / (idx+1)
                    precisions.append(precision_at_rank)
            ap_scores.append(np.mean(precisions))

    return np.mean(ap_scores)

def mrr(sims, unique_funcs, true_funcs):
    '''Mean Reciprocal Rank'''
    ranks = []

    for i, true_func in enumerate(true_funcs):
        sorted_indices = np.argsort(-sims[i])
        sorted_funcs = [unique_funcs[idx] for idx in sorted_indices]

        # Находим позицию правильной функции
        try:
            rank = sorted_funcs.index(true_func) + 1 
            ranks.append(1.0 / rank)
        except ValueError:
            ranks.append(0.0)

    return np.mean(ranks)

def rank_distribution(sims, unique_funcs, true_funcs):
    '''Распределение рангов'''
    ranks = []

    for i, true_func in enumerate(true_funcs):
        sorted_indices = np.argsort(-sims[i])
        sorted_funcs = [unique_funcs[idx] for idx in sorted_indices]

        try:
            rank = sorted_funcs.index(true_func) + 1
            ranks.append(rank)
        except ValueError:
            ranks.append(None)

    ranks = [r for r in ranks if r is not None]

    if ranks:
        print(f"Средний ранг: {np.mean(ranks):.2f}")
        print(f"Медианный ранг: {np.median(ranks):.2f}")
        print(f"Максимальный ранг: {np.max(ranks):.2f}")

    else:
        print("Нет найденных функций")

def metrics(model, df):
    """Основная функция метрик"""
    model.eval()
    unique_funcs = df['func'].unique().tolist()
    sims = get_similarity_matrix(model, df['pos_text'].tolist(), unique_funcs)
    true_funcs = df['func'].tolist()
    
    print()
    print(f"\033[36m{"—" * 30}\033[0m")
    for k in TOP_K_METRICS:
        print(f"Recall@{k}: {recall_at_k(sims, unique_funcs, true_funcs, k):.3f}")
        print(f"MAP@{k}: {map_at_k(sims, unique_funcs, true_funcs, k):.3f}")
        print("—" * 30)

    print(f"MRR: {mrr(sims, unique_funcs, true_funcs):.3f}")
    rank_distribution(sims, unique_funcs, true_funcs)
    print(f"\033[36m{"—" * 30}\033[0m")

### train function

In [ ]:
def semi_hard_mining(model, df, delta=0.2, device='cuda'):
    text_embeddings = model.encode(
        df['pos_text'].tolist(),
        convert_to_tensor=True,
        device=device
    )

    text_embeddings = F.normalize(text_embeddings, p=2, dim=1)
    sim_matrix = torch.mm(text_embeddings, text_embeddings.T)

    # Создаем маски
    funcs = df['func'].values
    n = len(df)

    # Хэшируем функции для быстрого сравнения
    funcs_tensor = torch.tensor([hash(f) for f in funcs], device=device)
    same_func_mask = funcs_tensor.unsqueeze(0) == funcs_tensor.unsqueeze(1)

    # Маска для себя
    self_mask = torch.eye(n, dtype=torch.bool, device=device)

    # Для каждого anchor находим positive (любой текст из той же функции, кроме себя)
    semi_hard_indices = []

    for i in tqdm(range(n), desc='make semi'):
        # Positive example (берем первый попавшийся из той же функции)
        pos_idx = torch.where(same_func_mask[i] & ~self_mask[i])[0]
        if len(pos_idx) == 0:
            semi_hard_indices.append(random.choice([j for j in range(n) if j != i]))
            continue

        pos_i = pos_idx[0].item()
        sim_ap = sim_matrix[i, pos_i].item()

        # Негативные кандидаты (из других функций)
        other_idx = torch.where(~same_func_mask[i])[0]
        other_sims = sim_matrix[i][other_idx]

        # Условие для semi-hard: sim_ap < sim_an < sim_ap + delta
        lower_bound = sim_ap
        upper_bound = sim_ap + delta

        semi_mask = (other_sims > lower_bound) & (other_sims < upper_bound)

        if semi_mask.any():
            # Случайный выбор из semi-hard
            candidates = other_idx[semi_mask]
            chosen = candidates[torch.randint(0, len(candidates), (1,))].item()
        else:
            valid_mask = other_sims < upper_bound
            if valid_mask.any():
                valid_sims = other_sims[valid_mask]
                valid_idx = other_idx[valid_mask]
                chosen = valid_idx[torch.argmax(valid_sims)].item()
            else:
                chosen = other_idx[torch.randint(0, len(other_idx), (1,))].item()

        semi_hard_indices.append(chosen)

    return [df['pos_text'].iloc[idx] for idx in semi_hard_indices]

In [ ]:
def mine_hard_negatives(epoch, model, df, top_k=10):
    text_embeddings = model.encode(
        df['pos_text'].tolist(),
        convert_to_tensor=True
    )
    text_embeddings = F.normalize(text_embeddings, p=2, dim=1)
    sim_matrix = torch.mm(text_embeddings, text_embeddings.T)

    # Создаем маску для других функций
    n = len(df)
    funcs = df['func'].tolist()

    same_func_mask = torch.zeros((n, n), dtype=torch.bool)
    for i in range(n):
        for j in range(n):
            if funcs[i] == funcs[j]:
                same_func_mask[i, j] = True

    self_mask = torch.eye(n, dtype=torch.bool)
    other_mask = ~same_func_mask & ~self_mask

    # Для каждого элемента берем топ-K из other_mask
    hard_indices = []
    for i in tqdm(range(n), desc='make hard'):
        # Получаем индексы других функций
        other_idx = torch.where(other_mask[i])[0]

        if len(other_idx) == 0:
            # Если нет других функций, берем все кроме себя
            other_idx = torch.where(~self_mask[i])[0]

        # Схожести с другими функциями
        other_sims = sim_matrix[i][other_idx]

        # Берем топ-K
        k = min(top_k, len(other_sims))
        top_indices = torch.topk(other_sims, k).indices

        # Выбираем индекс на основе эпохи
        chosen = other_idx[top_indices[epoch % k]].item()
        hard_indices.append(chosen)

    return [df['pos_text'].iloc[idx] for idx in hard_indices]

### validation function

In [ ]:
def metrics_new_texts(model, train_df, new_texts, new_true_funcs, k_list=TOP_K_METRICS):
    unique_funcs = train_df['func'].unique().tolist()
    sims = get_similarity_matrix(model, new_texts, unique_funcs)

    print()
    print(f"\033[36m{"—" * 30}\033[0m")
    for k in k_list:
        print(f"Recall@{k}: {recall_at_k(sims, unique_funcs, new_true_funcs, k):.3f}")
        print(f"MAP@{k}: {map_at_k(sims, unique_funcs, new_true_funcs, k):.3f}")
        print("—" * 30)

    print(f"MRR: {mrr(sims, unique_funcs, new_true_funcs):.3f}")
    rank_distribution(sims, unique_funcs, new_true_funcs)
    print(f"\033[36m{"—" * 30}\033[0m")

In [ ]:
def validation(model, train_df, val_df):
    model.eval()
    val_df['neg_text'] = np.roll(
        val_df['pos_text'], 
        shift=-(np.random.randint(len(val_df) // 2, len(val_df)) % len(val_df))
        )
    val_df['semi_hard'] = semi_hard_mining(model, val_df)
    val_df['hard_neg'] = mine_hard_negatives(1, model, val_df)

    experiment(model)
    delta_sim(model, val_df)
    metrics_new_texts(
        model,
        train_df, 
        val_df['pos_text'].tolist(),
        val_df['func'].tolist(),
        k_list=TOP_K_METRICS
    )

## Обучение

In [19]:
class TripletDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        return {
            "func": row["func"],
            "pos": row["pos_text"],
            "neg": row["neg_text"]
        }


In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def collate_fn(batch):
    funcs = [item["func"] for item in batch]
    poss = [item["pos"]  for item in batch]
    negs = [item["neg"]  for item in batch]

    func_enc = tokenizer(funcs, padding=True, truncation=True, return_tensors="pt")
    pos_enc = tokenizer(poss,  padding=True, truncation=True, return_tensors="pt")
    neg_enc = tokenizer(negs,  padding=True, truncation=True, return_tensors="pt")

    return {
        "func": func_enc,
        "pos": pos_enc,
        "neg": neg_enc
    }


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
class CustomSBERT(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_length = 256

        hidden = self.encoder.config.hidden_size

        self.bottleneck = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, hidden)
        )

    def freezer(self, n, freeze_head=False):
        for name, param in self.encoder.named_parameters():
            if "encoder.layer." in name:
                layer_num = int(name.split("encoder.layer.")[1].split(".")[0])
                param.requires_grad = (layer_num >= n)
            elif "embeddings" in name or "pooler" in name:
                param.requires_grad = (n < 24)

        for param in self.bottleneck.parameters():
            param.requires_grad = not freeze_head

        encoder_trainable = sum(p.numel() for p in self.encoder.parameters() if p.requires_grad)
        encoder_total = sum(p.numel() for p in self.encoder.parameters())
        head_trainable = sum(p.numel() for p in self.bottleneck.parameters() if p.requires_grad)

        print(f"Freezer: n={n}")
        print(f"  Encoder trainable: {encoder_trainable:,}/{encoder_total:,} ({encoder_trainable/encoder_total:.1%})")
        print(f"  Head trainable: {head_trainable:,}")
        print(f"  Total trainable: {encoder_trainable + head_trainable:,}")

    def pool(self, model_out, mask):
        token_embeddings = model_out[0]
        mask = mask.unsqueeze(-1)
        summed = torch.sum(token_embeddings * mask, dim=1)
        denom = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / denom

    def encode(
      self,
      texts,
      convert_to_tensor=False,
      device=DEVICE,
      batch_size=16,
      no_grad=True
  ):
      context = torch.no_grad() if no_grad else contextlib.nullcontext()

      with context:
          if isinstance(texts, dict) and "input_ids" in texts:
              batch = texts
              if device:
                  batch = {k: v.to(device) for k, v in batch.items()}
              out = self.encoder(**batch)
              pooled = self.pool(out, batch["attention_mask"])
              projected = self.bottleneck(pooled)
              return F.normalize(projected, p=2, dim=1)

          if isinstance(texts, str):
              texts = [texts]

          all_embeddings = []

          for i in range(0, len(texts), batch_size):
              chunk = texts[i:i+batch_size]

              batch = self.tokenizer(
                  chunk,
                  padding=True,
                  truncation=True,
                  max_length=self.max_length,
                  return_tensors="pt"
              )

              if device:
                  batch = {k: v.to(device) for k, v in batch.items()}

              out = self.encoder(**batch)
              pooled = self.pool(out, batch["attention_mask"])
              projected = self.bottleneck(pooled)
              emb = F.normalize(projected, p=2, dim=1)

              all_embeddings.append(emb.cpu())

          embeddings = torch.cat(all_embeddings, dim=0)

          if convert_to_tensor:
              return embeddings.to(device if device else "cpu")

          return embeddings.numpy()

    def forward(self, triplet_batch):
        func = self.encode(triplet_batch["func"], convert_to_tensor=True, no_grad=False)
        pos = self.encode(triplet_batch["pos"], convert_to_tensor=True, no_grad=False)
        neg = self.encode(triplet_batch["neg"], convert_to_tensor=True, no_grad=False)
        return func, pos, neg


In [23]:
def triplet_loss(a, p, n, margin=0.3):
    d_pos = 1 - F.cosine_similarity(a, p)
    d_neg = 1 - F.cosine_similarity(a, n)
    return torch.relu(d_pos - d_neg + margin).mean()

In [24]:
def train_one_epoch(model, loader, opt, device):
    model.train()
    total_loss = 0

    for batch in tqdm(loader, desc='train'):
        batch = {k: {kk: vv.to(device) for kk, vv in v.items()} for k, v in batch.items()}
        opt.zero_grad()

        func, pos, neg = model.forward(batch)
        loss = triplet_loss(func, pos, neg)

        loss.backward()
        opt.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def validate_epoch(model, loader, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in loader:
            batch = {k: {kk: vv.to(device) for kk, vv in v.items()} for k, v in batch.items()}
            func, pos, neg = model(batch)
            loss = triplet_loss(func, pos, neg)
            total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
def run_easy_neg_training(model, df, val_df, epochs, device, lr=2e-5):
    model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    for epoch in range(epochs):
        print(f"\n=== Easy Neg Epoch {epoch+1}/{epochs} ===")

        df["neg_text"] = np.roll(
            df["pos_text"],
            shift=np.random.randint(0, len(df))
        )

        dataset = TripletDataset(df)
        loader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

        loss = train_one_epoch(model, loader, opt, device)
        print(f"\033[33m train loss: {loss:.4f} \033[0m")

        val_dataset = TripletDataset(val_df)
        val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
        val_loss = validate_epoch(model, val_loader, device)
        print(f"\033[35m val loss: {val_loss:.4f} \033[0m")

        if (epoch + 1) % 5 == 0:
            experiment(model)
            metrics(model, df)

def run_semi_hard_training(model, df, val_df, epochs, device, lr=1e-5):
    model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    for epoch in range(epochs):
        print(f"\n=== Semi-Hard Epoch {epoch+1}/{epochs} ===")

        df["neg_text"] = semi_hard_mining(model, df)

        dataset = TripletDataset(df)
        loader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

        loss = train_one_epoch(model, loader, opt, device)
        print(f"\033[33m train loss: {loss:.4f} \033[0m")

        val_dataset = TripletDataset(val_df)
        val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
        val_loss = validate_epoch(model, val_loader, device)
        print(f"\033[35m val loss: {val_loss:.4f} \033[0m")

        if (epoch + 1) % 5 == 0:
            experiment(model)
            metrics(model, df)

def run_hard_neg_training(model, df, val_df, epochs, device, lr=1e-5, top_k=10):
    model.train()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)
    for epoch in range(epochs):
        print(f"\n=== Hard Neg Epoch {epoch+1}/{epochs} ===")

        df["neg_text"] = mine_hard_negatives(epoch, model, df, top_k=top_k)

        dataset = TripletDataset(df)
        loader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

        loss = train_one_epoch(model, loader, opt, device)
        print(f"\033[33m train loss: {loss:.4f} \033[0m")

        val_dataset = TripletDataset(val_df)
        val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
        val_loss = validate_epoch(model, val_loader, device)
        print(f"\033[35m val loss: {val_loss:.4f} \033[0m")

        if (epoch + 1) % 5 == 0:
            experiment(model)
            metrics(model, df)


## expiriens

In [26]:
train_df = pd.read_csv("train_data_split_train.csv")
val_df = pd.read_csv("train_data_split_test.csv")

----

In [ ]:
model = CustomSBERT(model_name=MODEL_NAME).to(DEVICE)


In [ ]:
metrics(model, train_df)
print('\n', '-'*25, 'VALIDATION', '-'*25)
validation(model, train_df, val_df)


——————————————————————————————
Recall@1: 0.166
MAP@1: 0.166
——————————————————————————————
Recall@10: 0.527
MAP@10: 0.266
——————————————————————————————
MRR: 0.281
Средний ранг: 36.82
Медианный ранг: 9.00
Максимальный ранг: 379.00
——————————————————————————————

 ------------------------- VALIDATION -------------------------


make hard: 100%|██████████| 1520/1520 [00:00<00:00, 3924.03it/s]



            pos: 0.520659863948822,
            neg: 0.5296279191970825
Средняя delta similarity для easy negatives: 0.39044964
Средняя delta similarity для hard negatives: 0.12442354

——————————————————————————————
Recall@1: 0.155
MAP@1: 0.155
——————————————————————————————
Recall@10: 0.484
MAP@10: 0.246
——————————————————————————————
MRR: 0.263
Средний ранг: 38.40
Медианный ранг: 12.00
Максимальный ранг: 364.00
——————————————————————————————


In [ ]:
model.freezer(24, freeze_head=False)
run_easy_neg_training(model, train_df, val_df, epochs=5, device=DEVICE, lr=1.5e-5)

print('\n', '-'*25, 'VALIDATION', '-'*25)
validation(model, train_df, val_df)

Freezer: n=24
  Encoder trainable: 0/426,908,672 (0.0%)
  Head trainable: 1,051,136
  Total trainable: 1,051,136

=== Easy Neg Epoch 1/5 ===


train: 100%|██████████| 974/974 [03:33<00:00,  4.57it/s]


 train loss: 0.0822 
 val loss: 0.0371 

=== Easy Neg Epoch 2/5 ===


train: 100%|██████████| 974/974 [03:33<00:00,  4.55it/s]


 train loss: 0.0473 
 val loss: 0.0323 

=== Easy Neg Epoch 3/5 ===


...

 train loss: 0.0325 
 val loss: 0.0258 

=== Easy Neg Epoch 5/5 ===


train: 100%|██████████| 974/974 [03:33<00:00,  4.57it/s]


 train loss: 0.0278 
 val loss: 0.0241 

            pos: 0.7487176060676575,
            neg: 0.22871966660022736

——————————————————————————————
Recall@1: 0.341
MAP@1: 0.341
——————————————————————————————
Recall@10: 0.794
MAP@10: 0.476
——————————————————————————————
MRR: 0.486
Средний ранг: 9.74
Медианный ранг: 3.00
Максимальный ранг: 375.00
——————————————————————————————

 ------------------------- VALIDATION -------------------------


make hard: 100%|██████████| 1520/1520 [00:00<00:00, 7274.47it/s]



            pos: 0.7487176060676575,
            neg: 0.22871966660022736
Средняя delta similarity для easy negatives: 0.96866786
Средняя delta similarity для hard negatives: 0.23250778

——————————————————————————————
Recall@1: 0.308
MAP@1: 0.308
——————————————————————————————
Recall@10: 0.753
MAP@10: 0.440
——————————————————————————————
MRR: 0.451
Средний ранг: 12.02
Медианный ранг: 3.00
Максимальный ранг: 334.00
——————————————————————————————


In [ ]:
model.freezer(20, freeze_head=True)
run_easy_neg_training(model, train_df, val_df, epochs=7, device=DEVICE, lr=2e-5)

print('\n', '-'*25, 'VALIDATION', '-'*25)
validation(model, train_df, val_df)

Freezer: n=20
  Encoder trainable: 174,984,192/426,908,672 (41.0%)
  Head trainable: 0
  Total trainable: 174,984,192

=== Easy Neg Epoch 1/7 ===


train: 100%|██████████| 974/974 [07:49<00:00,  2.08it/s]


 train loss: 0.0139 
 val loss: 0.0212 

=== Easy Neg Epoch 2/7 ===


train: 100%|██████████| 974/974 [07:48<00:00,  2.08it/s]


 train loss: 0.0110 
 val loss: 0.0186 

=== Easy Neg Epoch 3/7 ===


...

 train loss: 0.0069 
 val loss: 0.0123 

=== Easy Neg Epoch 7/7 ===


train: 100%|██████████| 974/974 [07:46<00:00,  2.09it/s]


 train loss: 0.0056 
 val loss: 0.0127 

 ------------------------- VALIDATION -------------------------


make hard: 100%|██████████| 1520/1520 [00:00<00:00, 6402.73it/s]



            pos: 0.9516693949699402,
            neg: -0.13880521059036255
Средняя delta similarity для easy negatives: 0.9563876
Средняя delta similarity для hard negatives: 0.16543123

——————————————————————————————
Recall@1: 0.443
MAP@1: 0.443
——————————————————————————————
Recall@10: 0.874
MAP@10: 0.579
——————————————————————————————
MRR: 0.585
Средний ранг: 6.70
Медианный ранг: 2.00
Максимальный ранг: 291.00
——————————————————————————————


In [ ]:
model.freezer(19, freeze_head=False)
run_easy_neg_training(model, train_df, val_df, epochs=10, device=DEVICE, lr=1.5e-5)

print('\n', '-'*25, 'VALIDATION', '-'*25)
validation(model, train_df, val_df)

Freezer: n=19
  Encoder trainable: 187,580,416/426,908,672 (43.9%)
  Head trainable: 1,051,136
  Total trainable: 188,631,552

=== Easy Neg Epoch 1/10 ===


train: 100%|██████████| 974/974 [07:58<00:00,  2.04it/s]


 train loss: 0.0050 
 val loss: 0.0154 

=== Easy Neg Epoch 2/10 ===


train: 100%|██████████| 974/974 [07:57<00:00,  2.04it/s]


 train loss: 0.0050 
 val loss: 0.0137 

=== Easy Neg Epoch 3/10 ===


...

 train loss: 0.0041 
 val loss: 0.0125 

=== Easy Neg Epoch 10/10 ===


train: 100%|██████████| 974/974 [07:58<00:00,  2.04it/s]


 train loss: 0.0034 
 val loss: 0.0128 

            pos: 0.7384870648384094,
            neg: 0.10534925758838654

——————————————————————————————
Recall@1: 0.555
MAP@1: 0.555
——————————————————————————————
Recall@10: 0.963
MAP@10: 0.693
——————————————————————————————
MRR: 0.696
Средний ранг: 2.80
Медианный ранг: 1.00
Максимальный ранг: 311.00
——————————————————————————————

 ------------------------- VALIDATION -------------------------


make hard: 100%|██████████| 1520/1520 [00:00<00:00, 6908.90it/s]



            pos: 0.7384870648384094,
            neg: 0.10534925758838654
Средняя delta similarity для easy negatives: 0.9942704
Средняя delta similarity для hard negatives: 0.15056027

——————————————————————————————
Recall@1: 0.458
MAP@1: 0.458
——————————————————————————————
Recall@10: 0.894
MAP@10: 0.601
——————————————————————————————
MRR: 0.607
Средний ранг: 5.96
Медианный ранг: 2.00
Максимальный ранг: 268.00
——————————————————————————————


In [ ]:
from safetensors.torch import save_file

model.eval()
state_dict = model.state_dict()
save_file(state_dict, "cust/custsbert-3-1.safetensors")
print('save')
experiment(model)

save

            pos: 0.7384870648384094,
            neg: 0.10534925758838654


----

In [ ]:
model = CustomSBERT(MODEL_NAME).to(DEVICE)
state_dict = load_file("cust/custsbert-3-1.safetensors")
model.load_state_dict(state_dict)
model.eval()
experiment(model)
print('\n', '-'*25, 'VALIDATION', '-'*25)
validation(model, train_df, val_df)


            pos: 0.7384870648384094,
            neg: 0.10534925758838654

 ------------------------- VALIDATION -------------------------


make hard: 100%|██████████| 1520/1520 [00:00<00:00, 5395.07it/s]



            pos: 0.7384870648384094,
            neg: 0.10534925758838654
Средняя delta similarity для easy negatives: 0.9893851
Средняя delta similarity для hard negatives: 0.15056027

——————————————————————————————
Recall@1: 0.458
MAP@1: 0.458
——————————————————————————————
Recall@10: 0.894
MAP@10: 0.601
——————————————————————————————
MRR: 0.607
Средний ранг: 5.96
Медианный ранг: 2.00
Максимальный ранг: 268.00
——————————————————————————————


In [ ]:
model.freezer(15, freeze_head=True)
run_semi_hard_training(model, train_df, val_df, epochs=15, device=DEVICE, lr=0.7e-5)

print('\n', '-'*25, 'VALIDATION', '-'*25)
validation(model, train_df, val_df)

Freezer: n=15
  Encoder trainable: 237,965,312/426,908,672 (55.7%)
  Head trainable: 0
  Total trainable: 237,965,312

=== Semi-Hard Epoch 1/15 ===


train: 100%|██████████| 974/974 [08:48<00:00,  1.84it/s]


 train loss: 0.1588 
 val lSoss: 0.0172 

=== Semi-Hard Epoch 2/15 ===


train: 100%|██████████| 974/974 [08:48<00:00,  1.84it/s]


 train loss: 0.1449 
 val lSoss: 0.0126 

=== Semi-Hard Epoch 3/15 ===


...

 train loss: 0.0325 
 val lSoss: 0.0201 

=== Semi-Hard Epoch 15/15 ===


train: 100%|██████████| 974/974 [08:38<00:00,  1.88it/s]


 train loss: 0.0338 
 val lSoss: 0.0380 

            pos: 0.631564736366272,
            neg: -0.022318854928016663

——————————————————————————————
Recall@1: 0.906
MAP@1: 0.906
——————————————————————————————
Recall@10: 0.982
MAP@10: 0.934
——————————————————————————————
MRR: 0.935
Средний ранг: 1.92
Медианный ранг: 1.00
Максимальный ранг: 334.00
——————————————————————————————

 ------------------------- VALIDATION -------------------------


make hard: 100%|██████████| 1520/1520 [00:00<00:00, 7294.73it/s]



            pos: 0.631564736366272,
            neg: -0.022318854928016663
Средняя delta similarity для easy negatives: 0.9966258
Средняя delta similarity для hard negatives: 0.36615947

——————————————————————————————
Recall@1: 0.582
MAP@1: 0.582
——————————————————————————————
Recall@10: 0.795
MAP@10: 0.650
——————————————————————————————
MRR: 0.656
Средний ранг: 19.49
Медианный ранг: 1.00
Максимальный ранг: 373.00
——————————————————————————————


In [ ]:
model.freezer(15, freeze_head=False)
run_semi_hard_training(model, train_df, val_df, epochs=5, device=DEVICE, lr=0.7e-5)

print('\n', '-'*25, 'VALIDATION', '-'*25)
validation(model, train_df, val_df)

Freezer: n=15
  Encoder trainable: 237,965,312/426,908,672 (55.7%)
  Head trainable: 1,051,136
  Total trainable: 239,016,448

=== Semi-Hard Epoch 1/5 ===


train: 100%|██████████| 974/974 [08:37<00:00,  1.88it/s]


 train loss: 0.0257 
 val lSoss: 0.0203 

=== Semi-Hard Epoch 2/5 ===


train: 100%|██████████| 974/974 [08:37<00:00,  1.88it/s]


 train loss: 0.0321 
 val lSoss: 0.0348 

=== Semi-Hard Epoch 3/5 ===


...

 train loss: 0.0240 
 val lSoss: 0.0359 

=== Semi-Hard Epoch 5/5 ===


train: 100%|██████████| 974/974 [08:32<00:00,  1.90it/s]


 train loss: 0.0194 
 val lSoss: 0.0208 

            pos: 0.5295838713645935,
            neg: 0.09958920627832413

——————————————————————————————
Recall@1: 0.959
MAP@1: 0.959
——————————————————————————————
Recall@10: 0.996
MAP@10: 0.973
——————————————————————————————
MRR: 0.973
Средний ранг: 1.28
Медианный ранг: 1.00
Максимальный ранг: 357.00
——————————————————————————————

 ------------------------- VALIDATION -------------------------


make hard: 100%|██████████| 1520/1520 [00:00<00:00, 4870.00it/s]



            pos: 0.5295838713645935,
            neg: 0.09958920627832413
Средняя delta similarity для easy negatives: 0.9982219
Средняя delta similarity для hard negatives: 0.32865018

——————————————————————————————
Recall@1: 0.626
MAP@1: 0.626
——————————————————————————————
Recall@10: 0.882
MAP@10: 0.715
——————————————————————————————
MRR: 0.719
Средний ранг: 10.03
Медианный ранг: 1.00
Максимальный ранг: 371.00
——————————————————————————————


In [ ]:
model.eval()
state_dict = model.state_dict()
save_file(state_dict, "cust/custsbert-3-2.safetensors")
print('save')
experiment(model)

save

            pos: 0.5295838713645935,
            neg: 0.09958920627832413


---

In [ ]:
model = CustomSBERT(MODEL_NAME).to(DEVICE)
state_dict = load_file("cust/custsbert-3-2.safetensors")
model.load_state_dict(state_dict)
model.eval()
experiment(model)
print('\n', '-'*25, 'VALIDATION', '-'*25)
validation(model, train_df, val_df)


            pos: 0.5295838713645935,
            neg: 0.09958920627832413

 ------------------------- VALIDATION -------------------------


make hard: 100%|██████████| 1520/1520 [00:00<00:00, 3761.60it/s]



            pos: 0.5295838713645935,
            neg: 0.09958920627832413
Средняя delta similarity для easy negatives: 0.98823106
Средняя delta similarity для hard negatives: 0.32865018

——————————————————————————————
Recall@1: 0.626
MAP@1: 0.626
——————————————————————————————
Recall@10: 0.882
MAP@10: 0.715
——————————————————————————————
MRR: 0.719
Средний ранг: 10.03
Медианный ранг: 1.00
Максимальный ранг: 371.00
——————————————————————————————


In [28]:
model.freezer(12, freeze_head=True)
run_hard_neg_training(model, train_df, val_df, epochs=10, device=DEVICE, lr=0.7e-5)

print('\n', '-'*25, 'VALIDATION', '-'*25)
validation(model, train_df, val_df)

Freezer: n=12
  Encoder trainable: 275,753,984/426,908,672 (64.6%)
  Head trainable: 0
  Total trainable: 275,753,984

=== Hard Neg Epoch 1/10 ===


train: 100%|██████████| 974/974 [08:29<00:00,  1.91it/s]


 train loss: 0.0224 
 val loss: 0.0379 

=== Hard Neg Epoch 2/10 ===


train: 100%|██████████| 974/974 [08:29<00:00,  1.91it/s]


 train loss: 0.0204 
 val loss: 0.0222 

=== Hard Neg Epoch 3/10 ===


...

 train loss: 0.0090 
 val loss: 0.0310 

=== Hard Neg Epoch 10/10 ===


train: 100%|██████████| 974/974 [08:28<00:00,  1.92it/s]


 train loss: 0.0071 
 val loss: 0.0215 

            pos: 0.7154603004455566,
            neg: 0.008061503991484642

——————————————————————————————
Recall@1: 0.993
MAP@1: 0.993
——————————————————————————————
Recall@10: 0.999
MAP@10: 0.996
——————————————————————————————
MRR: 0.996
Средний ранг: 1.06
Медианный ранг: 1.00
Максимальный ранг: 263.00
——————————————————————————————

 ------------------------- VALIDATION -------------------------


make hard: 100%|██████████| 1520/1520 [00:00<00:00, 4677.33it/s]



            pos: 0.7154603004455566,
            neg: 0.008061503991484642
Средняя delta similarity для easy negatives: 0.9992537
Средняя delta similarity для hard negatives: 0.36608773

——————————————————————————————
Recall@1: 0.650
MAP@1: 0.650
——————————————————————————————
Recall@10: 0.880
MAP@10: 0.735
——————————————————————————————
MRR: 0.739
Средний ранг: 9.80
Медианный ранг: 1.00
Максимальный ранг: 374.00
——————————————————————————————


----

In [ ]:
model.eval()
state_dict = model.state_dict()
save_file(state_dict, "cust/custsbert-3.safetensors")
print('save')
experiment(model)

save

            pos: 0.7154603004455566,
            neg: 0.008061503991484642
